# ORI-C Notebook 02 — Real Data Pilot: FRED Monthly

This notebook demonstrates the full ORI-C real-data workflow on the **FRED monthly pilot dataset**:

1. Validate proxy spec (ex ante gate)
2. Load and inspect the dataset
3. Run ORI-C pipeline via subprocess
4. Parse and visualise the causal test results
5. Interpret the verdict

---
**Prerequisite:** `conda activate cumulative_symbolic` from repo root.

In [ ]:
import sys
import json
import subprocess
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

# Repository root (one level up from examples/)
REPO_ROOT = Path('..').resolve()
sys.path.insert(0, str(REPO_ROOT / 'src'))
sys.path.insert(0, str(REPO_ROOT / '04_Code'))

print(f'Repo root: {REPO_ROOT}')

%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

## 1. Validate the proxy spec

Before any run, we validate the `proxy_spec.json` — this is the **ex ante gate**.
A broken spec (wrong column name, invalid normalization) fails here, not silently downstream.

In [ ]:
SPEC_PATH = REPO_ROOT / '03_Data/real/fred_monthly/proxy_spec.json'
CSV_PATH  = REPO_ROOT / '03_Data/real/fred_monthly/real.csv'

result = subprocess.run(
    [sys.executable,
     str(REPO_ROOT / '04_Code/pipeline/validate_proxy_spec.py'),
     '--spec', str(SPEC_PATH),
     '--csv',  str(CSV_PATH)],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('VALIDATION ERRORS:', result.stderr)
else:
    print('Proxy spec is valid — proceeding.')

## 2. Load and inspect the dataset

In [ ]:
df_raw = pd.read_csv(CSV_PATH)
print(f'Shape: {df_raw.shape}')
print(f'Columns: {list(df_raw.columns)}')
df_raw.head()

In [ ]:
# Basic descriptive stats
df_raw.describe()

In [ ]:
# Visual overview of O, R, I proxies
fig, axes = plt.subplots(3, 1, figsize=(12, 7), sharex=True)
for ax, col, color in zip(axes,
                          ['O', 'R', 'I'],
                          ['#2563eb', '#0d9488', '#7c3aed']):
    if col in df_raw.columns:
        ax.plot(df_raw.index, df_raw[col], color=color, lw=1.2)
        ax.set_ylabel(col, fontsize=11, fontweight='bold')
        ax.axhline(df_raw[col].mean(), color='gray', lw=0.8, ls='--', alpha=0.6)

axes[-1].set_xlabel('Time index')
fig.suptitle('FRED Monthly — O, R, I proxies', fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Run ORI-C pipeline

We call `run_real_data_demo.py` as a subprocess so all outputs (logs, tables, figures)
are written to the output directory as they would be in production CI.

In [ ]:
OUT_DIR = REPO_ROOT / '05_Results/nb02_fred_pilot'
OUT_DIR.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable,
    str(REPO_ROOT / '04_Code/pipeline/run_real_data_demo.py'),
    '--input',        str(CSV_PATH),
    '--outdir',       str(OUT_DIR),
    '--time-mode',    'index',
    '--normalize',    'robust_minmax',
    '--control-mode', 'no_symbolic',
    '--seed',         '123',
]

print('Running:', ' '.join(cmd[-6:]), '...')
result = subprocess.run(cmd, capture_output=True, text=True, cwd=str(REPO_ROOT))
print(result.stdout[-2000:] if len(result.stdout) > 2000 else result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr[-1000:])

## 4. Read run outputs

In [ ]:
summary_path = OUT_DIR / 'tables/summary.json'
verdict_path = OUT_DIR / 'verdict.txt'

if summary_path.exists():
    with open(summary_path) as f:
        summary = json.load(f)
    print('=== summary.json ===')
    for k, v in summary.items():
        print(f'  {k}: {v}')

if verdict_path.exists():
    print('\n=== verdict.txt ===')
    print(verdict_path.read_text().strip())

In [ ]:
ts_path = OUT_DIR / 'tables/test_timeseries.csv'
if ts_path.exists():
    df_ts = pd.read_csv(ts_path)
    print(f'Time series shape: {df_ts.shape}')
    df_ts[['t', 'O', 'R', 'I', 'Cap', 'S', 'C', 'delta_C']].head()

## 5. Visualise C(t) and ΔC(t)

In [ ]:
if 'df_ts' in dir() and not df_ts.empty:
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

    ax1.plot(df_ts['t'], df_ts['C'], color='#7c3aed', lw=1.5, label='C(t)')
    ax1.axhline(0, color='gray', lw=0.8, ls='--')
    ax1.set_ylabel('C(t) — Order variable')
    ax1.legend(fontsize=9)

    ax2.plot(df_ts['t'], df_ts['delta_C'], color='#0d9488', lw=1.2, label='ΔC(t)')
    ax2.axhline(0, color='gray', lw=0.8, ls='--')
    ax2.set_ylabel('ΔC(t)')
    ax2.set_xlabel('Time index')
    ax2.legend(fontsize=9)

    fig.suptitle('FRED Monthly — C(t) and ΔC(t)', fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('Time series not available — check that the run completed successfully.')

## 6. Run causal tests (Granger, VAR, bootstrap CI)

In [ ]:
causal_cmd = [
    sys.executable,
    str(REPO_ROOT / '04_Code/pipeline/tests_causaux.py'),
    '--run-dir',     str(OUT_DIR),
    '--alpha',       '0.01',
    '--lags',        '1-5',
    '--pre-horizon', '50',
    '--post-horizon','50',
    '--seed',        '123',
]

print('Running causal tests...')
result_c = subprocess.run(causal_cmd, capture_output=True, text=True, cwd=str(REPO_ROOT))
print(result_c.stdout[-3000:] if len(result_c.stdout) > 3000 else result_c.stdout)
if result_c.returncode != 0:
    print('STDERR:', result_c.stderr[-1000:])

In [ ]:
verdict_json_path = OUT_DIR / 'tables/verdict.json'
if verdict_json_path.exists():
    with open(verdict_json_path) as f:
        verdict_data = json.load(f)

    print('=== Causal Test Verdict ===')
    print(f'  verdict:           {verdict_data.get("verdict", "N/A")}')
    print(f'  binary_detected:   {verdict_data.get("binary_detected", "N/A")}')
    print(f'  ok_p:              {verdict_data.get("ok_p", "N/A")}  (source: {verdict_data.get("ok_p_source", "N/A")})')
    print(f'  p_shift (Welch):   {verdict_data.get("p_shift", "N/A")}')
    print(f'  boot_lo / boot_hi: {verdict_data.get("boot_lo", "N/A")} / {verdict_data.get("boot_hi", "N/A")}')
    print(f'  min_granger:       {verdict_data.get("min_granger_s_to_dc", "N/A")}')
    print(f'  coint_p:           {verdict_data.get("coint_p", "N/A")}')

## 7. Interpretation guide

| Verdict | Meaning |
|---------|--------|
| `seuil_detecte` | Threshold detected; Granger + bootstrap CI support S→C causal claim |
| `non_detecte` | No threshold detected (pre-threshold regime) |
| `falsifie` | False positive in pre-window OR C(t) level below minimum |
| `indetermine_sigma_nul` | Σ(t) ≡ 0 post-threshold; symbolic canal inoperable |
| `indetermine_stats_indisponibles` | All p-sources NaN — series too short for statistics |

**Decision rule (pre-registered):** α = 0.01, 99% CI, SESOI = +0.3 robust SD via MAD.

Null results are expected and reported equally. See `ORIC_POINT_OF_TRUTH.md` for the full normative framework.

---
**Next:** see `notebook_03_robustness_analysis.ipynb` for sensitivity analysis across parameter variants.